<div align="center">

# Bring fse to your existing tools

Use one local encoder with your search, topic, and table libraries.

</div>

## When do you need this?

You already use a search library, a topic model, or a table library.
You want it to use the **same local encoder** as the rest of your app.
Choose a section below after running the shared setup, rather than installing a separate model for every tool.

## The story in pictures

<picture>
<source media="(prefers-color-scheme: dark)" srcset="media/06-integrations-dark.svg">
<img src="media/06-integrations-light.svg" alt="One POTION encoder supplies vectors to search libraries, a topic model, and a table column.">
</picture>

## 1. Prepare one encoder and real articles

The examples share POTION-base-8M and 100 [AG News](https://huggingface.co/datasets/SetFit/ag_news) articles.
**Embeddings** are vectors, lists of numbers that represent texts.
Each library receives those vectors explicitly, using the pinned versions from the [index](README.md#run-a-recipe).

In [ ]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

In [ ]:
import fse

encoder = recipe.encoder()
texts = recipe.news(12)
print(texts[0])
print("Vector dimensions:", encoder.output_dim)

In [ ]:
texts = recipe.news(100)

def embed(items):
    return encoder.encode(list(items), workers=1)

query = texts[0]
print("Example article:", query)

## 2. Retrieve passages with DSPy

A **retriever** returns texts related to a query.
DSPy can call our local encoder directly, without a language model or a paid API.
We query with an existing article to **check the connection** before trying new search phrases.

In [ ]:
import dspy

retriever = dspy.retrievers.Embeddings(corpus=texts, embedder=embed, k=1)
result = retriever(query)
assert result.passages == [texts[0]]
print("DSPy found:", result.passages[0])

## 3. Group articles with BERTopic

**Topic modeling** groups documents and finds words that describe each group.
We provide vectors, use PCA to reduce their dimensions, then ask KMeans for four groups.
These are demonstration settings, so **inspect the topic words** before treating them as useful categories.

In [ ]:
from bertopic import BERTopic
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

topic_model = BERTopic(
    embedding_model=None, umap_model=PCA(n_components=5),
    hdbscan_model=KMeans(n_clusters=4, random_state=7, n_init=10),
    vectorizer_model=CountVectorizer(stop_words="english"), calculate_probabilities=False,
)
topics, _ = topic_model.fit_transform(texts, embeddings=embed(texts))
assert len(topics) == len(texts) and len(set(topics)) == 4
print(topic_model.get_topic_info()[["Topic", "Count", "Name"]].to_string(index=False))

## 4. Search with txtai

An **external transform** tells txtai to call your encoder instead of loading its own.
The callable accepts a batch of texts and returns one vector per text.

In [ ]:
from txtai import Embeddings
index = Embeddings({"method": "external", "transform": embed, "content": True})
index.index([(i, text, None) for i, text in enumerate(texts)])
hits = index.search(query, 1)
assert hits[0]["id"] == "0" or hits[0]["id"] == 0
print("txtai:", hits)
index.close()

## 5. Store vectors in Chroma

Pass vectors when adding documents and when searching.
**Disable the default embedding function** so Chroma does not download another model.

In [ ]:
import chromadb
from chromadb.config import Settings
chroma = chromadb.EphemeralClient(Settings(anonymized_telemetry=False))
collection = chroma.create_collection("fse-recipe", embedding_function=None)
collection.add(ids=[str(i) for i in range(len(texts))], documents=texts, embeddings=embed(texts).tolist())
hits = collection.query(query_embeddings=embed([query]).tolist(), n_results=1)
assert hits["ids"] == [["0"]]
print("Chroma:", hits["ids"])

## 6. Search a local LanceDB table

Keep each article next to its vector in a temporary local table.
The query uses the same encoder, so stored and query vectors have the same meaning.

In [ ]:
import lancedb
lance = lancedb.connect(str(work / "lance"))
table = lance.create_table("texts", [{"id": i, "text": text, "vector": row.tolist()} for i, (text, row) in enumerate(zip(texts, embed(texts)))])
hits = table.search(embed([query])[0]).limit(1).to_list()
assert hits[0]["id"] == 0
print("LanceDB:", hits[0]["text"])

## 7. Write batches to Qdrant

A **sink** accepts vectors as fse produces them.
This small application adapter allocates a batch buffer, then upserts completed rows into local Qdrant.
Stable row IDs make replay replace existing points.
Memory-only Qdrant cannot promise crash durability, so this adapter reports no `durable_rows`.

In [ ]:
from qdrant_client import QdrantClient, models
client = QdrantClient(":memory:")
client.create_collection("texts", vectors_config=models.VectorParams(size=encoder.output_dim, distance=models.Distance.COSINE))

class QdrantSink:
    def __init__(self, client, collection, dim):
        self.client, self.collection, self.dim = client, collection, dim
        self.next_row = 0
    def target(self, first_row, n_rows):
        return np.empty((n_rows, self.dim), dtype=np.float32)
    def commit(self, first_row, rows):
        if first_row != self.next_row:
            raise ValueError("commits must be contiguous")
        self.client.upsert(self.collection, points=[models.PointStruct(id=first_row+i, vector=row.tolist()) for i, row in enumerate(rows)], wait=True)
        self.next_row += len(rows)

sink: fse.Sink = QdrantSink(client, "texts", encoder.output_dim)
for batch in encoder.encode_batches([texts[:50], texts[50:]], sink=sink, workers=1):
    assert batch.durable_rows is None
assert client.count("texts", exact=True).count == len(texts)
print("Qdrant sink points:", client.count("texts", exact=True).count)
client.close()

## 8. Add vectors to a Polars table

A Polars **Series** is one column.
fse reads the text column through Arrow, the public table interchange interface, and returns a vector matrix.
Add it as a fixed-size array column, rather than making one Python call per row.

In [ ]:
import polars as pl
frame = pl.DataFrame({"text": texts})
rows = encoder.encode(frame["text"], workers=1)
np.testing.assert_array_equal(rows, embed(texts))
frame = frame.with_columns(pl.Series("vector", rows, dtype=pl.Array(pl.Float32, encoder.output_dim)))
assert frame.height == len(texts)
print(frame.head(3))

## Result

The search examples return the article used as the query, and BERTopic reports four groups.
The assertions check that each library receives and stores **the intended rows**.
This proves these connections work for this example, rather than measuring search quality.

## What next?

- [Handle queue replay before copying to Qdrant](04-kafka-and-qdrant.ipynb).
- [fse API and design](../docs/architecture/runtime.md)
- [DSPy retriever](https://github.com/stanfordnlp/dspy/blob/main/dspy/retrievers/embeddings.py), [BERTopic vectors](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html), [txtai configuration](https://neuml.github.io/txtai/embeddings/configuration/vectors/), [Chroma](https://docs.trychroma.com/docs/embeddings/embedding-functions), [LanceDB](https://docs.lancedb.com/search/vector-search), [Qdrant](https://github.com/qdrant/qdrant-client), [Polars](https://docs.pola.rs/user-guide/expressions/user-defined-python-functions/)

Back to the [recipe index](README.md) or the [main README](../README.md#recipes-from-a-model-to-a-working-application).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [ ]:
recipe.close()